# Mastermind

Donea Fernado-Emanuel

grupa 243

In [7]:
import random
from copy import deepcopy

ISTORIC_INCERCARI=[]
COD_SECRET=[]


def evaluare(incercare, secret):
    piese_ok=0 #piese puse corect
    cul_ok=0 #culoare corecta dar pozitie gresit

    viz_secret=[0,0,0,0,0]
    viz_incercare=[0,0,0,0,0]

    #numaram piesele care se afla pe pozitia corecta
    for i in range(5):
        if incercare[i]==secret[i]:
            piese_ok+=1
            viz_secret[i]=1
            viz_incercare[i]=1

    #numraam piesele care au culoarea corecta dar pe poz incorecte
    for i in range(5):
        if viz_incercare[i]==0: #piese care nu se afla pe pozitia corecta

            #parcurgem secretele care nu sunt inca marcate
            for j in range(5):
                if viz_secret[j]==0 and incercare[i]==secret[j]:
                    cul_ok+=1
                    viz_secret[j]=1
                    break #ne oprim
    return piese_ok,cul_ok


def make_candidate(): #face un candidat random
    return [random.randint(0, 9) for _ in range(5)]


def fitness(candidate):
    # un-fitness, adica scor cat mai mic => buun
    global ISTORIC_INCERCARI
    scor=0

    for element in ISTORIC_INCERCARI:
        incercare_veche=element[0]
        feedback_real=element[1]

        feedback_simulat = evaluare(incercare_veche, candidate) #daca candidatul ar fi codul secret, ce evaluare ar prima incercarea veche

       #diferenta cat mai mica=>good
        dif_piese=abs(feedback_real[0]-feedback_simulat[0])
        dif_cul=abs(feedback_real[1]-feedback_simulat[1])

        scor=scor+ dif_piese+dif_cul

    return scor




**Justifcare un-fitness**

Am ales o functia de "un-fitness", in care un scor cat mai apropiat de zero indica un candidat ideal

Evaluarea-ul este reprezentat de
- **numarul de piese puse corect**
- **numarul de cate piese sunt de culoare corecta dar pe pozitii incorecte**.

Pentru fiecare candidat, se parcurc incercarile vechi, iar algoritmul calculeaza diferenta in modul dintre **evaluarea reala**(primit de joc) si **evaluarea pe care ar gener-o candidatul daca el ar fi codul secret**. Suma acestor diferente reprezinta scorul final.


Cromozomii care sunt pe aprope primesc o penalizare mica si sunt selectati mai departe, in loc sa fie catalogati direct "gresit" sau "corect". Deci fitnesul populatiei evolueaza catre zero


In [8]:

def mutation(candidate): #perturbare random
    mutant=deepcopy(candidate)

    index_cromzom=random.randint(0,4)
    mutatie_cromozom=random.randint(0,9)

    mutant[index_cromzom]=mutatie_cromozom
    return mutant

def crossover(parent1, parent2): #facem niste interschimbari
    m=random.randint(1,3)
    copil1=parent1[:m]+parent2[m:]
    copil2=parent2[:m]+parent1[m:]

    return copil1,copil2

def full_genetic_alg(dimensiune_populatie, max_generatii, k_elita):
    #faceti populatia initiala, printati top candidates si fitnessurile lor, selectia(pastrati k% din cei mai buni), apoi repopulati(crossover&mutation)

    #cream populatie initiala
    populatie=[make_candidate() for _ in range(dimensiune_populatie)]

    for generatie in range(max_generatii):

        #evaluam fitnesul
        lista_fitness=[(individ, fitness(individ)) for individ in populatie]
        lista_fitness.sort(key=lambda x:x[1])#sortam dupa fitness

        #verificam daca am gasit codul din prima
        if lista_fitness[0][1]==0:
            return lista_fitness[0][0]

        #pastram elitele
        elita=[individ for individ, scor in lista_fitness[:k_elita]]

        #repopularea
        populatie_noua=elita.copy()
        while len(populatie_noua) < dimensiune_populatie:
            p1=random.choice(elita)
            p2=random.choice(elita)
            c1,c2=crossover(p1,p2)

            if random.random() <0.1:
                c1=mutation(c1)
            if random.random() <0.1:
                c2=mutation(c2)

            populatie_noua.append(c1)
            populatie_noua.append(c2)

        populatie=populatie_noua[:dimensiune_populatie]


    lista_fitness=[(individ,fitness(individ)) for individ in populatie]
    lista_fitness.sort(key=lambda x:x[1])
    return lista_fitness[0][0] #returnam cel mai bun individ


def mastermind():
    global COD_SECRET
    global ISTORIC_INCERCARI

    COD_SECRET=make_candidate()
    ISTORIC_INCERCARI=[]

    print(f"Cheia generata este {COD_SECRET}")

    for tura in range(20):
        if tura==0:
            incercare=make_candidate() #prima data ghicim la intmplare
        else:
            incercare=full_genetic_alg(dimensiune_populatie=150, max_generatii=50, k_elita=20)

        #obtinem feedback din joc
        rezultat=evaluare(incercare, COD_SECRET)
        ISTORIC_INCERCARI.append((incercare, rezultat))

        if rezultat[0]==5: #5 piese corecte
            print(f"Cheia a fost ghicita in {tura+1} incercari!")
            break

    for guess, scor in ISTORIC_INCERCARI:
        print(f"{guess}")
        print(f"Piese pe pozitia corecta {scor[0]} | Culori corecte {scor[1]}")
        print()






In [9]:
mastermind()

Cheia generata este [7, 5, 9, 7, 8]
Cheia a fost ghicita in 7 incercari!
[2, 8, 2, 1, 5]
Piese pe pozitia corecta 0 | Culori corecte 2

[1, 3, 1, 8, 3]
Piese pe pozitia corecta 0 | Culori corecte 1

[7, 1, 9, 9, 2]
Piese pe pozitia corecta 2 | Culori corecte 0

[8, 0, 8, 9, 2]
Piese pe pozitia corecta 0 | Culori corecte 2

[5, 1, 9, 4, 0]
Piese pe pozitia corecta 1 | Culori corecte 1

[7, 2, 9, 5, 7]
Piese pe pozitia corecta 2 | Culori corecte 2

[7, 5, 9, 7, 8]
Piese pe pozitia corecta 5 | Culori corecte 0

